
# 2 - Erstellung der info.json

Dieses Script erstellt eine info.json Datei für alle Objekte der Collection und legt sie im Ordner 'info' als JSON Datei ab.
Doku der Info.json, siehe GOCFL-implementierung von Jürgen Enge, https://github.com/je4/gocfl . 
Vorlage: https://github.com/je4/gocfl/blob/main/gocfl-info-1.0.json 


## Export info.json

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/{signature}.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins working directory als Inventar, welche Datenobjekte eingelagert wurden. 



## Variante A: Input-File aus Alma 

Standardverfahren: Die meisten collections der ZHB können mit einer Eingabedatei verarbeitet werden. Eine Excel-Datei liegt im working directory. Sie kann relativ leicht aus Alma exportiert werden. Die Sammlungen der Sosa sind alle in Alma in einem öffentlichen Set in der RZS gelistet. 

Die Export-Datei wurde leicht überarbeitet. Nicht benötigte Spalten werden gelöscht, einige Daten müssen gesplitted werden. Folgende Spalten werden benötigt:

- Title
- Record number: wird vorerst nicht benötigt, kann trotzdem stehengelassen werden.
- Call number: aus Spalte Availability splitten, Spalte umbenennen 
- MMS_ID als Text erzwingen (Bsp. '9914249335105505')
- DOI manuell ergänzen
- Dateipfad manuell ergänzen
- externe ID wie z.B. E-Manuscripta ID manuell ergänzen

Da die Sosa-Sammlungen der ZHB i.d.R. überschaubar sind, hält sich der zeitliche Aufwand dafür in Grenzen.


In [ ]:
import json
import pandas as pd
import config

from datetime import datetime


# needed variables: files, paths, input

input_file = config.input_file
collection = config.collection_id
info_dir = f'{collection}/{config.info_path}/'
urn = config.ingest_workflow
org_id = config.organisation_id
coll_id = config.collection_id
fulljsonfile = f'{config.inventory_file}'
fullexcelfile = f'{config.inventory_xlsx}'

completeSet = []

today = datetime.today().strftime('%Y-%m-%d')
now = datetime.now().isoformat()
doc_counter = 0

# Read the Excel file into a pandas DataFrame
df = pd.read_excel(input_file)

for _, row in df.iterrows():

    doc_counter += 1
    infoSet = {
    # infoset created after https://github.com/je4/gocfl/blob/main/gocfl-info-1.0.json 

        "signature": "",
        "organisation_id": org_id,
        "organisation": config.organisation,
        "organisation_address": config.organisation_address,
        "collection_id": coll_id,
        "collection": config.collection,
        "sets": config.sets,
        "identifiers": [],
        "title": "",
        "alternative_titles": [],
        "description": "",
        "keywords": config.keywords,
        "user": config.user_name,
        "address": config.user_address,
        "created": now,
        "last_changed": now,
        "deprecates": "",
        "references": [],
        "ingest_workflow": urn,
        "additional": ""
    }
    
    # identifiers
        
    doi = row['DOI']
    mms_id = str(row['MMS ID'])
    callnumber = row['Call number']
    title = row['Title']
    sip_path = row['Dateipfad']
    external_id = str(row['externe ID'])
    
    # folder name and signature:
    foldername = doi.replace('.','_').replace('/','_')
    signature = f'{org_id}:{coll_id}_{foldername}'
    
    # references
    doiurl = config.baseurl_doi+doi
    almaurl = config.baseurl_alma+mms_id
    
    #complete info.json
    
    infoSet["identifiers"] = ['doi:'+doi, 'mmsid:'+mms_id, org_id+':'+callnumber, urn+':'+external_id]
    infoSet["references"] = [doiurl, almaurl, foldername]
    infoSet["signature"] = signature
    infoSet["title"] = title
    infoSet["additional"] = sip_path.replace('\\','/')

    #print(infoSet)
    
    completeSet.append(infoSet)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    infofile = f"{info_dir}{foldername}.json"

    with open(infofile, "w") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
        

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)

with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nAll JSON written to {fulljsonfile}")
    
# Writing completeSet as Excel file

df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"All data saved to Excel file as {fullexcelfile}")
print(f"Total records: {doc_counter}\nFinished at {now}")




## Variante B: Input-File aus Zenodo 

Standardverfahren für alle Repository-Daten (Lory, Lara): 
Diese collections der ZHB können ebenfalls mit einer Eingabedatei verarbeitet werden.  Sie kann mit dem folgenden Script aus Zenodo exportiert werden. Danach liegt eine Excel-Datei im working directory.



In [ ]:
import requests
import pandas as pd
import datetime 
import config

community = config.collection_id
output = f'{community}.xlsx'
zenodoRestUrl = "https://zenodo.org/api/records"
headers = {}
headers["Content-Type"] = "application/json"
size = '100'
params = { "communities":  community, "size": size}


def getRecords():
    r = requests.get(f"{zenodoRestUrl}", params=params, headers=headers)
    return r.json()

result = getRecords()
numOfRec = (result["hits"]["total"])
print(f"Number of Records in Community {community}: {numOfRec}")

localRecordCounter = 1
remotePaginator = 1
resultSet = []
while localRecordCounter < int(numOfRec):
    params["page"] = remotePaginator
    result = getRecords()
    for record in result["hits"]["hits"]:
        resultDet = {}
        resultDet["created"] = record["updated"]
        resultDet["doi"] = record["doi"]
        resultDet["zenodo_id"] = record["recid"]
        resultDet["reference"] = record["doi_url"]
        resultDet["title"]= record["metadata"]["title"]
        resultDet["filepath"]= record["links"]["files"]
        

        print(f"#{localRecordCounter}: {record['doi']}")

        localRecordCounter += 1
        resultSet.append(resultDet)
    remotePaginator += 1
    print(f"go to next {size} records page {remotePaginator}")

print(f"Number of Records writing to file: {localRecordCounter}")

#Schreibe Metadaten (resultSet) in ein XLS-File zur Weiterbearbeitung
df = pd.DataFrame(resultSet) 
#print(df.head())
df.to_excel(output) 
print("Finished at:", datetime.datetime.now())

## Variante B, Teil 2

Nun wird die Infojson erstellt, ähnlich wie oben. 

In [1]:
import json
import pandas as pd
import config
from datetime import datetime

# needed variables: files, paths, input

input_file = config.input_file
collection = config.collection_id
info_dir = f'{collection}/{config.info_path}/'
urn = config.ingest_workflow
org_id = config.organisation_id
coll_id = config.collection_id
fulljsonfile = f'{config.inventory_file}'
fullexcelfile = f'{config.inventory_xlsx}'

completeSet = []

today = datetime.today().strftime('%Y-%m-%d')
now = datetime.now().isoformat()
doc_counter = 0

# Read the Excel file into a pandas DataFrame
df = pd.read_excel(input_file)

for _, row in df.iterrows():

    doc_counter += 1
    infoSet = { 

        "signature": "",
        "organisation_id": org_id,
        "organisation": config.organisation,
        "organisation_address": config.organisation_address,
        "collection_id": coll_id,
        "collection": config.collection,
        "sets": config.sets,
        "identifiers": [],
        "title": row['title'],
        "alternative_titles": [],
        "description": "",
        "keywords": config.keywords,
        "user": config.user_name,
        "address": config.user_address,
        "created": row["created"],
        "last_changed": now,
        "deprecates": "",
        "references": row["reference"],
        "ingest_workflow": urn,
        "additional": row["filepath"]
    }
    
    # identifiers
        
    doi = row['doi']
    zenodo_id = str(row['zenodo_id'])
    
    # folder name and signature:
    foldername = doi.replace('.','_').replace('/','_')
    signature = f'{org_id}:{coll_id}_{foldername}'
        
    #complete info.json    
    infoSet["identifiers"] = ['doi:'+doi, 'zenodo:'+zenodo_id]
    infoSet["signature"] = signature

    #print(infoSet)    
    completeSet.append(infoSet)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    infofile = f"{info_dir}{foldername}.json"

    with open(infofile, "w", encoding="utf-8") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
        

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)

with open(fulljsonfile, "w", encoding="utf-8") as outfile:
    outfile.write(fulldump)
    print(f"---\nAll JSON written to {fulljsonfile}")
    
# Writing completeSet as Excel file

df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"All data saved to Excel file as {fullexcelfile}")
print(f"Total records: {doc_counter}\nFinished at {now}")




info.json saved as lory_unilu/info/10_5281_zenodo_10466776.json
info.json saved as lory_unilu/info/10_5281_zenodo_10471407.json
info.json saved as lory_unilu/info/10_5281_zenodo_10471212.json
info.json saved as lory_unilu/info/10_5281_zenodo_10471210.json
info.json saved as lory_unilu/info/10_5281_zenodo_10471153.json
info.json saved as lory_unilu/info/10_5281_zenodo_10470760.json
info.json saved as lory_unilu/info/10_5281_zenodo_10470627.json
info.json saved as lory_unilu/info/10_5281_zenodo_10469478.json
info.json saved as lory_unilu/info/10_5281_zenodo_10468315.json
info.json saved as lory_unilu/info/10_5281_zenodo_10137907.json
info.json saved as lory_unilu/info/10_5281_zenodo_10407260.json
info.json saved as lory_unilu/info/10_5281_zenodo_10407165.json
info.json saved as lory_unilu/info/10_5281_zenodo_10407163.json
info.json saved as lory_unilu/info/10_5281_zenodo_10405457.json
info.json saved as lory_unilu/info/10_5281_zenodo_10401674.json
info.json saved as lory_unilu/info/10_52

info.json saved as lory_unilu/info/10_5281_zenodo_7575811.json
info.json saved as lory_unilu/info/10_5281_zenodo_7590291.json
info.json saved as lory_unilu/info/10_5281_zenodo_7541651.json
info.json saved as lory_unilu/info/10_5281_zenodo_7510436.json
info.json saved as lory_unilu/info/10_5281_zenodo_7476880.json
info.json saved as lory_unilu/info/10_5281_zenodo_7476319.json
info.json saved as lory_unilu/info/10_5281_zenodo_7438220.json
info.json saved as lory_unilu/info/10_5281_zenodo_7410413.json
info.json saved as lory_unilu/info/10_5281_zenodo_7410347.json
info.json saved as lory_unilu/info/10_5281_zenodo_7389691.json
info.json saved as lory_unilu/info/10_5281_zenodo_7383057.json
info.json saved as lory_unilu/info/10_5281_zenodo_7382808.json
info.json saved as lory_unilu/info/10_5281_zenodo_7382806.json
info.json saved as lory_unilu/info/10_5281_zenodo_7261006.json
info.json saved as lory_unilu/info/10_5281_zenodo_6998787.json
info.json saved as lory_unilu/info/10_5281_zenodo_73471

info.json saved as lory_unilu/info/10_5281_zenodo_1305983.json
info.json saved as lory_unilu/info/10_5281_zenodo_1306529.json
info.json saved as lory_unilu/info/10_5281_zenodo_1308803.json
info.json saved as lory_unilu/info/10_5281_zenodo_1308870.json
info.json saved as lory_unilu/info/10_5281_zenodo_5566829.json
info.json saved as lory_unilu/info/10_14361_9783839459638.json
info.json saved as lory_unilu/info/10_5281_zenodo_4030998.json
info.json saved as lory_unilu/info/10_5281_zenodo_4030981.json
info.json saved as lory_unilu/info/10_5281_zenodo_4031003.json
info.json saved as lory_unilu/info/10_5281_zenodo_4031007.json
info.json saved as lory_unilu/info/10_5281_zenodo_4031001.json
info.json saved as lory_unilu/info/10_5281_zenodo_4030991.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305878.json
info.json saved as lory_unilu/info/10_5281_zenodo_1306464.json
info.json saved as lory_unilu/info/10_5281_zenodo_3238246.json
info.json saved as lory_unilu/info/10_5281_zenodo_13071

info.json saved as lory_unilu/info/10_5281_zenodo_1305866.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305864.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305868.json
info.json saved as lory_unilu/info/10_5281_zenodo_1310711.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305876.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305859.json
info.json saved as lory_unilu/info/10_5281_zenodo_1310706.json
info.json saved as lory_unilu/info/10_5281_zenodo_1310704.json
info.json saved as lory_unilu/info/10_5281_zenodo_1306057.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305993.json
info.json saved as lory_unilu/info/10_5281_zenodo_1306050.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305896.json
info.json saved as lory_unilu/info/10_5281_zenodo_1305880.json
info.json saved as lory_unilu/info/10_5281_zenodo_1306144.json
info.json saved as lory_unilu/info/10_5281_zenodo_1306154.json
info.json saved as lory_unilu/info/10_5281_zenodo_13065

info.json saved as lory_unilu/info/10_5281_zenodo_3764717.json
info.json saved as lory_unilu/info/10_5281_zenodo_3822609.json
info.json saved as lory_unilu/info/10_5281_zenodo_3820692.json
info.json saved as lory_unilu/info/10_5281_zenodo_3814747.json
info.json saved as lory_unilu/info/10_5281_zenodo_3801693.json
info.json saved as lory_unilu/info/10_5281_zenodo_3793562.json
info.json saved as lory_unilu/info/10_5281_zenodo_3793493.json
info.json saved as lory_unilu/info/10_5281_zenodo_3742056.json
info.json saved as lory_unilu/info/10_5281_zenodo_3742054.json
info.json saved as lory_unilu/info/10_5281_zenodo_3732166.json
info.json saved as lory_unilu/info/10_5281_zenodo_3597347.json
info.json saved as lory_unilu/info/10_5281_zenodo_3716138.json
info.json saved as lory_unilu/info/10_5281_zenodo_3698000.json
info.json saved as lory_unilu/info/10_5281_zenodo_3665685.json
info.json saved as lory_unilu/info/10_5281_zenodo_3695398.json
info.json saved as lory_unilu/info/10_5281_zenodo_36953

info.json saved as lory_unilu/info/10_5281_zenodo_2557538.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557458.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557455.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557443.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557439.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557415.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557398.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557384.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557372.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557368.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557366.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557362.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557352.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557326.json
info.json saved as lory_unilu/info/10_5281_zenodo_2557312.json
info.json saved as lory_unilu/info/10_5281_zenodo_25573

info.json saved as lory_unilu/info/10_5281_zenodo_1284067.json
info.json saved as lory_unilu/info/10_5281_zenodo_1284051.json
info.json saved as lory_unilu/info/10_5281_zenodo_1283775.json
info.json saved as lory_unilu/info/10_5281_zenodo_1256081.json
info.json saved as lory_unilu/info/10_5281_zenodo_1256011.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255934.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255918.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255916.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255914.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255912.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255910.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255906.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255829.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255806.json
info.json saved as lory_unilu/info/10_5281_zenodo_1255604.json
info.json saved as lory_unilu/info/10_5281_zenodo_12555